# [Step 5 - WebBaseLoader] Pulling a live web page

**MLCourse - Agentic AI - Module 05: Document Loaders**

> Stage in the capstone: stage 1 INGEST: the capstone reads user-supplied documents with exactly these loaders

## What you'll learn

- how `WebBaseLoader` fetches HTML and extracts visible text via BeautifulSoup
- what lands in metadata (`title`, `language`, `source`, `description`)
- a real manipulation: stripping short boilerplate lines from the payload
- web etiquette: internet requirements, robots.txt, throttling, JS limits

---

In [1]:
# =====================================================================
# CELL 1 - SHARED SETUP: imports, track discovery, download-once cache
# =====================================================================

# --- Standard library -------------------------------------------------
import os                      # file-system odds and ends
import urllib.request          # kept for parity with sibling notebooks
from pathlib import Path       # object-oriented filesystem paths

# --- Lesson-specific imports ------------------------------------------
from langchain_community.document_loaders import WebBaseLoader

# ---------------------------------------------------------------------
# TRACK WALKER - resolve 03_agentic_ai by walking upward from cwd.
# ---------------------------------------------------------------------
def _find_track(start: Path) -> Path:
    """Return the absolute path of the 03_agentic_ai track root."""
    for candidate in (start, *start.parents):
        hit = candidate / "03_agentic_ai"
        if hit.is_dir():
            return hit.resolve()
    raise FileNotFoundError(
        f"No directory named 03_agentic_ai found above {start} - "
        "run this notebook from somewhere inside the MLCourse repo."
    )

TRACK = _find_track(Path.cwd())
DATA = TRACK / "data"
DATA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# DOWNLOAD-ONCE HELPERS - present for consistency; today's specimen loads
# live through WebBaseLoader instead (that IS the lesson).
# ---------------------------------------------------------------------
def get_bytes(fname: str, url: str) -> bytes:
    """Return the file's bytes, downloading only on the very first call."""
    target = DATA / fname
    if target.exists() and target.stat().st_size > 0:
        payload = target.read_bytes()
        print(f"[cache] {fname}: {len(payload):,} bytes")
        return payload
    print(f"[fetch] {url}")
    request = urllib.request.Request(url, headers={"User-Agent": "MLCourse/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = response.read()
    target.write_bytes(payload)
    print(f"[saved] {fname}: {len(payload):,} bytes")
    return payload

def get_text(fname: str, url: str, encoding: str = "utf-8-sig") -> str:
    """get_bytes + decode; drops a BOM if present."""
    return get_bytes(fname, url).decode(encoding, errors="replace")

def to_ascii(text: str) -> str:
    """Console-safe printing for arbitrary text."""
    return text.encode("ascii", errors="replace").decode("ascii")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

C:\Users\Thoyajaksha Kashyap\AppData\Local\Temp\ipykernel_67736\3560564630.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


USER_AGENT environment variable not set, consider setting it to identify your requests.


## 1. Loading FROM THE WEB - different contract than files

Files sit still; websites change between runs, belong to someone, and bury
content under navigation menus. `WebBaseLoader` therefore:

1. HTTP-fetches the URL (needs INTERNET - nothing loads without it);
2. parses HTML with BeautifulSoup and extracts visible text;
3. returns one Document per URL with rich metadata:
   `title`, `language`, `description`, plus `source` = the URL itself.

Etiquette baked into this lesson: we throttle with `requests_per_second`,
and we treat `robots.txt` as binding. Politeness is a feature, not a chore.

Specimen: Wikipedia's *Machine learning* article - static HTML, generous,
and famously full of navigation boilerplate (perfect for the cleaning demo).

In [2]:
WIKI_URL = "https://en.wikipedia.org/wiki/Machine_learning"

loader = WebBaseLoader(WIKI_URL, requests_per_second=1)   # <= be polite
docs = loader.load()

print(f"Documents : {len(docs)}")                 # one per URL
meta = docs[0].metadata
for key in ("source", "title", "language", "description"):
    value = meta.get(key, "<missing>")
    print(f"{key:>12} : {to_ascii(str(value))[:90]}")

raw_text = docs[0].page_content
print(f"\npage_content : {len(raw_text):,} characters")

Documents : 1
      source : https://en.wikipedia.org/wiki/Machine_learning
       title : Machine learning - Wikipedia
    language : en
 description : <missing>

page_content : 132,110 characters


## 2. Inspect: the good, the bad, the boilerplate

BeautifulSoup hands us ALL visible text - article body AND menu entries,
footer links, edit buttons. Scroll the preview below and spot the noise;
then we measure how much of the payload is actually prose.

In [3]:
lines = [ln.strip() for ln in raw_text.splitlines()]
non_empty = [ln for ln in lines if ln]

print("--- first 15 non-empty lines (spot the boilerplate!) ---")
for line in non_empty[:15]:
    print(f"  | {to_ascii(line)[:76]}")

short_lines = [ln for ln in non_empty if len(ln) < 40]
share_short = 100.0 * len(short_lines) / max(len(non_empty), 1)
print(f"\nlines shorter than 40 chars : {len(short_lines)} "
      f"of {len(non_empty)} ({share_short:.1f}%) <- mostly UI chrome")

--- first 15 non-empty lines (spot the boilerplate!) ---
  | Machine learning - Wikipedia
  | Jump to content
  | Main menu
  | Main menu
  | move to sidebar
  | hide
  | Navigation
  | Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us
  | Contribute
  | HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages
  | Search
  | Search
  | Appearance
  | Donate
  | Create account

lines shorter than 40 chars : 1083 of 1493 (72.5%) <- mostly UI chrome


## 3. One meaningful manipulation: keep substantial lines

Real article prose lives in long paragraphs; navigation survives in short
fragments. Filtering lines under 40 characters removes most chrome without
touching the body - a deliberately simple heuristic you can refine later
(readability scores, tag-aware extraction, trafilatura-style tools).

In [4]:
MIN_LINE_CHARS = 40
kept_lines = [ln for ln in non_empty if len(ln) >= MIN_LINE_CHARS]
cleaned_text = "\n".join(kept_lines)

removed = len(raw_text) - len(cleaned_text)
reduction_pct = 100.0 * removed / max(len(raw_text), 1)
print(f"before : {len(raw_text):>9,} chars")
print(f"after  : {len(cleaned_text):>9,} chars  ({reduction_pct:.1f}% removed)")
print(f"lines  : {len(non_empty):>6} -> {len(kept_lines)}")

print("\n--- cleaned opening ---")
print(to_ascii(cleaned_text[:400]))

# Keep provenance honest: record the transformation in metadata.
clean_doc = type(docs[0])(
    page_content=cleaned_text,
    metadata={
        **meta,
        "cleaning": f"dropped lines shorter than {MIN_LINE_CHARS} chars",
    },
)
print("\ncleaned metadata['cleaning']:", clean_doc.metadata["cleaning"])

before :   132,110 chars
after  :   114,086 chars  (13.6% removed)
lines  :   1493 -> 410

--- cleaned opening ---
Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us
HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages
Toggle Relationships to other fields subsection
Proprietary software with free and open-source editions
Afrikaans?????????????????????Az?rbaycanca?????????????????????????????????????????????????????BosanskiCatal???????e?tinaCymraegDanskDeutsch????????Esper

cleaned metadata['cleaning']: dropped lines shorter than 40 chars


## 4. Pitfalls worth remembering

**Pitfall - internet is mandatory**: reruns hit the network again (and the
page may have CHANGED). Unlike our download-once helpers, WebBaseLoader has
no cache - for stable corpora, save `page_content` to disk after loading.

**Pitfall - JavaScript-rendered sites come back nearly EMPTY**: BeautifulSoup
sees server-sent HTML only. SPAs (React/Vue apps) need browser-based loaders
instead - check `len(page_content)` before celebrating.

**Pitfall - scraping is privileged access**: respect `robots.txt`, rate-limit,
prefer official APIs when offered, and identify yourself via headers.

**Pro-tip**: `WebBaseLoader` accepts a LIST of URLs and parallelises politely
- bulk ingestion of doc sites becomes one call.

## Takeaway

**WebBaseLoader = fetch + BeautifulSoup + one Document per URL with
title/language metadata. It hands you EVERYTHING visible - so budget a
cleaning step, throttle your hits, and cache results you depend on.**

## Summary

- Wikipedia's ML article arrived as one Document whose metadata names the
  page (`title`, `language='en'`, `source`=URL).
- Roughly a third of raw lines were sub-40-character chrome; the length
  filter removed that noise while preserving the article body.
- Metadata stayed intact through cleaning - transformations must never
  destroy provenance.
- No internet, no load: plan caching for reproducible pipelines.